In [ ]:
import itertools
import os

import cmasher as cm
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import scipy


In [ ]:
folder = r"J:\ctgroup\Edward\DATA\VMI\20250227\xe_2,5W\calibrated"

data = pd.DataFrame()
for file in sorted(os.listdir(folder)):
    if file.endswith('.h5'):
        print(file)
        d = pd.read_hdf(os.path.join(folder, file))
        d['angle'] = float(file.split('_')[0]) - 2
        d = d[d['pz'] > 0]
        d = d[(d['raw_t'] > 200) & (d['raw_t'] < 240)]
        d['px'] = d['px'] - 0.0015

        # rotate in x-z plane by angle degrees
        angle = np.radians(d['angle'].iloc[0] + 5)
        x_new = d['px'] * np.cos(angle) - d['pz'] * np.sin(angle)
        z_new = d['px'] * np.sin(angle) + d['pz'] * np.cos(angle)
        d['px'] = x_new
        d['pz'] = z_new
        d['py'] = d['py'] + 0.009
        data = pd.concat([data, d], ignore_index=True)
        d['px'] = -d['px']
        # d['py']=-d['py']
        d['pz'] = -d['pz']
        data = pd.concat([data, d], ignore_index=True)

data['pr'] = np.sqrt(data['px'] ** 2 + data['py'] ** 2 + data['pz'] ** 2)

data = data[data['pr'] < 0.8]

In [ ]:
for l in range(0, 5):
    for m in range(-l, l + 1):
        data[f'Y{l}{m}'] = scipy.special.sph_harm(m, l, np.arctan2(data['pz'], data['px']),
                                                  np.arccos(data['py'] / data['pr']))
for l in range(0, 5):
    for m in range(-l, l + 1):
        if m == 0:
            data[f'Y{l}{m}_real'] = data[f'Y{l}{m}'].to_numpy().real
        elif m > 0:
            data[f'Y{l}{m}_real'] = np.sqrt(2) * data[f'Y{l}{m}'].to_numpy().real
            data[f'Y{l}{-m}_real'] = np.sqrt(2) * data[f'Y{l}{m}'].to_numpy().imag

means = data.groupby('angle').mean().reset_index()
means_count = data.groupby('angle').count().reset_index()
means_std = data.groupby('angle').std().reset_index()
means_err = means_std / means_count ** 0.5

In [ ]:


sp = data[data['angle'] == 40]
sm = data[data['angle'] == -40]
spf = len(sp[sp['py'] > 0])
smf = len(sm[sm['py'] > 0])
spb = len(sp[sp['py'] < 0])
smb = len(sm[sm['py'] < 0])

PECD = 2 * ((spf - spb) / (spf + smf) - (smf - smb) / (smf + smb))
print(PECD)

# data = data.sample(frac=0.1)

In [ ]:
prs = []
pecds = []
signals = []
b10p = []
b10m = []
for bin in itertools.pairwise(np.linspace(0, 0.8, 50)):
    sp_bin = sp[(sp['pr'] > bin[0]) & (sp['pr'] <= bin[1])]
    sm_bin = sm[(sm['pr'] > bin[0]) & (sm['pr'] <= bin[1])]
    spf = len(sp_bin[sp_bin['py'] > 0])
    smf = len(sm_bin[sm_bin['py'] > 0])
    spb = len(sp_bin[sp_bin['py'] < 0])
    smb = len(sm_bin[sm_bin['py'] < 0])
    signal = spf + spb + smf + smb
    if signal > 2000:
        pecds.append(
                2 * ((spf - spb) / (spf + spf) - (smf - smb) / (smf + smb))
        )
        prs.append((bin[0] + bin[1]) / 2)
        signals.append(spf + spb + smf + smb)
        b10p.append(sp_bin['Y30_real'].mean())
        b10m.append(sm_bin['Y30_real'].mean())


In [ ]:
#create figure with secondary y axis
fig = go.Figure(
        data=[
            go.Bar(
                    x=prs,
                    y=signals,
                    name='Signal',
                    yaxis='y2',
                    marker_color='lightgray',
                    opacity=0.5,
            ),
            go.Scatter(
                    x=prs,
                    y=pecds,
                    name='PECD',
                    mode='lines+markers',
                    marker_color='blue',
            ),
        ],
        layout=go.Layout(
                title='Momentum Resolved PECD for Xe',
                xaxis=dict(title='Momentum (a.u.)'),
                yaxis=dict(title='PECD', range=[-0.5, 0.5]),
                yaxis2=dict(
                        title='Signal (counts)',
                        overlaying='y',
                        side='right',
                        showgrid=False,
                ),
                legend=dict(x=0.85, y=0.95),
                height=600,
                width=800,
        ),
)
fig.show()

In [ ]:
# create figure with secondary y axis for b10

fig = go.Figure(
        data=[
            go.Bar(
                    x=prs,
                    y=signals,
                    name='Signal',
                    yaxis='y2',
                    marker_color='lightgray',
                    opacity=0.5,
            ),
            go.Scatter(
                    x=prs,
                    y=b10p,
                    name='b10 (Positive Angle)',
                    mode='lines+markers',
                    marker_color='red',
            ),
            go.Scatter(
                    x=prs,
                    y=b10m,
                    name='b10 (Negative Angle)',
                    mode='lines+markers',
                    marker_color='green',
            ),
        ],
        layout=go.Layout(
                title='Momentum Resolved b10 for Propylene Oxide',
                xaxis=dict(title='Momentum (a.u.)'),
                yaxis=dict(title='b10 (a.u.)'),
                yaxis2=dict(
                        title='Signal (counts)',
                        overlaying='y',
                        side='right',
                        showgrid=False,
                ),
                legend=dict(x=0.75, y=0.95),
                height=600,
                width=800,
        ),
)
fig.show()

In [ ]:
data2 = data.copy().sort_values(by=['angle', 'pr'])


In [ ]:
bins = np.linspace(0, 0.8, 100)
data2['pr_bin'] = pd.cut(data2['pr'], bins=bins)

forward_df = data2[data2['py'] > 0]
backward_df = data2[data2['py'] < 0]

hist_df = data2.groupby(['pr_bin', 'angle']).size().reset_index()
hist_df['pr'] = hist_df['pr_bin'].apply(lambda x: x.mid)

px.scatter(
        hist_df,
        x="pr",
        y="angle",
        color=0,
        height=300,
        width=800,
        color_continuous_scale=px.colors.sequential.Inferno,
).update_layout(
        template="plotly_dark",
)

In [ ]:
hist_df_forward = forward_df[np.abs(forward_df['pz']) < 0.1].groupby(['pr_bin', 'angle']).size()
hist_df_backward = backward_df[np.abs(backward_df['pz']) < 0.1].groupby(['pr_bin', 'angle']).size()

diff_df = (2 * (hist_df_forward - hist_df_backward) / (hist_df_forward + hist_df_backward)).reset_index()
diff_df['pr'] = hist_df['pr'] = hist_df['pr_bin'].apply(lambda x: x.mid)
diff_df['counts'] = (hist_df_forward + hist_df_backward).reset_index()[0]

px.scatter(
        diff_df,
        x="pr",
        y="angle",
        color=0,
        size="counts",
        height=600,
        width=800,
        color_continuous_scale=px.colors.diverging.RdBu,
        color_continuous_midpoint=0,
        template="plotly_dark",
        range_color=[-0.2, 0.2],
).update_layout(
        coloraxis_colorbar_title="Relative FBA"
)



In [ ]:
K
data2['py_c'] = data2['py'] - 0.009
forward_df = data2[data2['py_c'] > 0]
backward_df = data2[data2['py_c'] < 0]

thin_ds = pd.concat([forward_df[np.abs(forward_df['pz']) < 0.1], backward_df[np.abs(backward_df['pz']) < 0.1]])
thin_ds = thin_ds[thin_ds['pr'] < 0.6]

forward_df_filt = forward_df[(np.abs(forward_df['pz']) < 0.1) & (forward_df['pr'] < 0.6) & (forward_df['pr'] > 0.15)]
backward_df_filt = backward_df[
    (np.abs(backward_df['pz']) < 0.1) & (backward_df['pr'] < 0.6) & (backward_df['pr'] > 0.15)]

FBA = 2 * (len(forward_df_filt) - len(backward_df_filt)) / (len(forward_df_filt) + len(backward_df_filt))
image, xe, ye = np.histogram2d(
        thin_ds[thin_ds['angle'] == 0]['px'],
        thin_ds[thin_ds['angle'] == 0]['py_c'],
        bins=128,
        range=[(-0.6, 0.6), (-0.6, 0.6)],
)

xc = xe[:-1]
yc = ye[:-1]
px.imshow(
        np.log(image.T),
        x=xc,
        y=yc,
        width=800,
        height=800,
).add_hline(
        y=0,
        line_color='black',
        line_width=1,
        line_dash='dot',
).add_vline(
        x=0,
        line_color='black',
        line_width=1,
        line_dash='dot',
).add_annotation(
        text=f"FBA = {FBA * 100:.2f}%",
        x=0,
        y=0,

).show()

px.histogram(thin_ds[thin_ds['angle'] == 0], ['pr'])

px.imshow(
        (image - image[:, ::-1]).T,
        x=xc,
        y=yc,
        width=800,
        height=800,
        color_continuous_midpoint=0,
        range_color=[-40, 40]
).add_hline(
        y=0,
        line_color='black',
        line_width=1,
        line_dash='dot',
).add_vline(
        x=0,
        line_color='black',
        line_width=1,
        line_dash='dot',
).show()